# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. The dataset is defined by a Croissant schema and provides ordered logistic regression outputs for variables affecting knowledge adoption in rangeland management among pastoral households in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. This step downloads metadata (including the Croissant schema) and prepares the dataset for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"Dataset: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview

Review available record sets and their IDs, along with their fields. All elements are referenced by their unique `@id`.

We print the available record sets and their field IDs.

In [ ]:
# List all record sets and their fields using their '@id'.
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    print("Available Record Sets:\n")
    for rs in dataset.metadata.record_sets:
        print(f"  Record Set name: {rs.name}, @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for f in rs.fields:
                field_type = getattr(f, 'data_type', None)
                print(f"      - {getattr(f, 'name', f.id)} (ID: {f.id}, type: {field_type})")
        print()
else:
    print("No record sets found in metadata! Please check the Croissant schema or dataset availability.")

## 3. Data Extraction

Load data from each available record set into pandas DataFrames for further analysis. Each record set and field is referenced by their `@id`.

In [ ]:
# Extract data for all available record sets (referenced by their @id)
dataframes = dict()
record_set_ids = []

if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        rs_id = rs.id
        record_set_ids.append(rs_id)
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
    
    print(f"Loaded record sets (by @id): {record_set_ids}\n")
    primary_record_set_id = record_set_ids[0]
    print(f"Columns in record set '{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets found to extract data from.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, or grouping. All fields should be referenced by their `@id`.

In [ ]:
# Example: Filter by a numeric field (referenced by its @id), then normalize it.
# Replace the below example field @ids as needed based on the overview output above.

from numpy import number

# Choose primary record set as target
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

# Select candidate numeric field by @id
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    threshold = df[numeric_field_id].mean()  # for demonstration; could use another threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    
    print(f"Filtered records with '{numeric_field_id}' > {threshold:.3f} (using field @id):")
    display(filtered_df.head())
    
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' (field @id) for filtered records:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())
    
    # Attempt groupby using another field if available
    potential_group_fields = [col for col in df.columns if (col not in numeric_fields and df[col].nunique() < 20)]
    if potential_group_fields:
        group_field_id = potential_group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(f"\nGrouped data by '{group_field_id}' (field @id), showing mean {numeric_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA in this record set.")

## 5. Visualization

Visualize data distributions and relationships between fields. All identifiers use the exact `@id` as discussed above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot for a numeric field (by @id)
if numeric_fields:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (by @id)")
    plt.xlabel(numeric_field_id)
    
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of '{numeric_field_id}' (by @id)")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # Scatter by group if such a field exists:
    if potential_group_fields:
        group_field_id = potential_group_fields[0]
        plt.figure(figsize=(8, 5))
        sns.violinplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library, referencing all elements by their `@id`. We reviewed record sets and fields, extracted data into pandas DataFrames, and performed introductory analysis (filtering, normalization, grouping, and visualization) while referring only to the identifiers specified in the Croissant metadata. This provides a reproducible and extensible basis for deeper modeling and FAIR-compliant analysis of knowledge adoption predictors in rangeland management in Kenya.